# Training Pipeline: Master Router + Per-Daerah Isolation Forest
Notebook ini melatih satu Isolation Forest per `nama_daerah`, lalu menggabungkannya ke dalam satu `master_model.joblib`.


## 1. Library, Path, and Configuration

In [1]:
import json
import re
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cleaned_data_path = Path('../data/cleaned/cleaned_data.csv')
cleaned_train_dir = Path('../data/cleaned/train')
cleaned_test_dir = Path('../data/cleaned/test')
artifact_root = Path('../artifacts/post_award_anomaly')
region_root = artifact_root / 'by_daerah'
master_bundle_path = artifact_root / 'master_model.joblib'
master_manifest_path = artifact_root / 'master_model_manifest.json'

train_ratio = 0.85
contamination = 0.03
random_state = 42
min_rows_per_region = 100

numeric_features = [
    'tender_minvalue', 'award_value', 'days_to_award', 'budget_utilization_ratio', 'value_gap',
    'log_tender_minvalue', 'log_award_value', 'title_word_count', 'award_title_word_count',
    'supplier_count', 'award_value_per_day', 'same_day_award_flag', 'award_month', 'award_quarter', 'award_weekday'
]
categorical_features = ['mainprocurementcategory']
base_export_columns = [
    'lspe_id', 'nama_daerah', 'date', 'buyer_name', 'tender_title', 'mainprocurementcategory',
    'tender_minvalue', 'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier',
    'days_to_award', 'budget_utilization_ratio'
]

for output_dir in [cleaned_train_dir, cleaned_test_dir, artifact_root]:
    if output_dir.exists():
        for item in output_dir.iterdir():
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
    output_dir.mkdir(parents=True, exist_ok=True)

region_root.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'Train ratio         : {train_ratio}')
print(f'Contamination       : {contamination}')
print(f'Min rows per daerah : {min_rows_per_region}')
print(f'SHAP version        : {shap.__version__}')
print('Output folders      : reset on every run')


Configuration loaded.
Train ratio         : 0.85
Contamination       : 0.03
Min rows per daerah : 100
SHAP version        : 0.51.0
Output folders      : reset on every run


## 2. Helper Function and Data Loading

In [2]:
def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).strip().lower()).strip('_')

def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_builtin(v) for v in value]
    if isinstance(value, tuple):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value

def format_number(value):
    if pd.isna(value):
        return 'missing'
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        return f'{value:,.4f}'.rstrip('0').rstrip('.')
    return str(value)

def assign_severity(scores, medium_cutoff, anomaly_threshold):
    return np.select([scores >= anomaly_threshold, scores >= medium_cutoff], ['high', 'medium'], default='low')

def engineer_features(frame):
    data = frame.copy()
    data['date'] = pd.to_datetime(data['date'], errors='coerce')
    data['award_date'] = pd.to_datetime(data['award_date'], errors='coerce')
    data['award_month'] = data['award_date'].dt.month
    data['award_quarter'] = data['award_date'].dt.quarter
    data['award_weekday'] = data['award_date'].dt.weekday
    data['log_tender_minvalue'] = np.log1p(data['tender_minvalue'].clip(lower=0))
    data['log_award_value'] = np.log1p(data['award_value'].clip(lower=0))
    data['value_gap'] = data['award_value'] - data['tender_minvalue']
    data['title_word_count'] = data['tender_title'].fillna('').astype(str).str.split().str.len()
    data['award_title_word_count'] = data['award_title'].fillna('').astype(str).str.split().str.len()
    supplier_tokens = data['award_supplier'].fillna('').astype(str).str.split(',')
    data['supplier_count'] = supplier_tokens.apply(lambda items: max(len([item for item in items if str(item).strip()]), 1))
    safe_days = data['days_to_award'].replace(0, 1)
    data['award_value_per_day'] = data['award_value'] / safe_days
    data['same_day_award_flag'] = (data['days_to_award'] == 0).astype(int)
    return data

def make_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', encoder)]), categorical_features),
    ])

def clean_feature_names(names):
    cleaned = []
    for name in names:
        if name.startswith('num__'):
            cleaned.append(name.replace('num__', ''))
        elif name.startswith('cat__mainprocurementcategory_'):
            cleaned.append('cat_' + name.replace('cat__mainprocurementcategory_', ''))
        else:
            cleaned.append(name)
    return cleaned

def aggregate_shap_values(row_shap, feature_names_clean):
    aggregated = {}
    for feature_name, shap_value in zip(feature_names_clean, row_shap):
        raw_feature = 'mainprocurementcategory' if feature_name.startswith('cat_') else feature_name
        aggregated[raw_feature] = aggregated.get(raw_feature, 0.0) + float(shap_value)
    return aggregated

def make_scored_output(frame, scores, medium_cutoff, anomaly_threshold):
    output = frame.reset_index(drop=True).copy()
    output['anomaly_score'] = scores
    output['severity_band'] = assign_severity(scores, medium_cutoff, anomaly_threshold)
    output['prediction_label'] = np.select([scores >= anomaly_threshold, scores >= medium_cutoff], ['anomaly', 'warning'], default='normal')
    output['anomaly_flag'] = (scores >= anomaly_threshold).astype(int)
    return output

def build_local_explanations(frame, shap_values, feature_names_clean):
    output = frame.reset_index(drop=True).copy()
    for rank in [1, 2, 3]:
        output[f'top_{rank}_feature'] = None
        output[f'top_{rank}_shap'] = np.nan
        output[f'top_{rank}_impact'] = None
    explanations = []
    for row_idx in range(len(output)):
        aggregated = aggregate_shap_values(shap_values[row_idx], feature_names_clean)
        top_features = sorted(aggregated.items(), key=lambda item: abs(item[1]), reverse=True)[:3]
        parts = []
        for rank, (feature_name, shap_value) in enumerate(top_features, start=1):
            raw_value = output.loc[row_idx, feature_name] if feature_name in output.columns else output.loc[row_idx, 'mainprocurementcategory']
            impact = 'pushes anomaly score up' if shap_value > 0 else 'pushes anomaly score down'
            output.loc[row_idx, f'top_{rank}_feature'] = feature_name
            output.loc[row_idx, f'top_{rank}_shap'] = float(shap_value)
            output.loc[row_idx, f'top_{rank}_impact'] = impact
            parts.append(f"{feature_name}={format_number(raw_value)} ({impact}, SHAP {float(shap_value):.4f})")
        explanations.append(f"[{output.loc[row_idx, 'severity_band'].upper()}] " + '; '.join(parts))
    output['human_readable_explanation'] = explanations
    return output

def build_reference_thresholds(train_output):
    category_reference = train_output.groupby('mainprocurementcategory').agg(
        award_value_p99=('award_value', lambda series: float(series.quantile(0.99))),
        days_to_award_p95=('days_to_award', lambda series: float(series.quantile(0.95))),
        budget_ratio_p05=('budget_utilization_ratio', lambda series: float(series.quantile(0.05))),
        budget_ratio_p95=('budget_utilization_ratio', lambda series: float(series.quantile(0.95)))
    ).reset_index()
    global_reference = {
        'award_value_p99': float(train_output['award_value'].quantile(0.99)),
        'days_to_award_p95': float(train_output['days_to_award'].quantile(0.95)),
        'budget_ratio_p05': float(train_output['budget_utilization_ratio'].quantile(0.05)),
        'budget_ratio_p95': float(train_output['budget_utilization_ratio'].quantile(0.95))
    }
    return {'category_reference': category_reference.to_dict(orient='records'), 'global_reference': global_reference}, category_reference

def select_demo_cases(test_output):
    picked_frames = []
    for label in ['normal', 'warning', 'anomaly']:
        subset = test_output.loc[test_output['prediction_label'] == label].copy()
        if subset.empty:
            continue
        if label == 'anomaly':
            chosen = subset.sort_values('anomaly_score', ascending=False).head(1)
        else:
            subset['distance_to_median'] = (subset['anomaly_score'] - subset['anomaly_score'].median()).abs()
            chosen = subset.sort_values('distance_to_median').head(1).drop(columns=['distance_to_median'])
        picked_frames.append(chosen)
    if not picked_frames:
        return test_output.head(0).copy()
    return pd.concat(picked_frames, ignore_index=True)

def make_time_based_split(region_data, train_ratio):
    date_counts = (
        region_data.groupby('award_date')
            .size()
            .rename('row_count')
            .reset_index()
            .sort_values('award_date')
            .reset_index(drop=True)
    )

    if len(date_counts) < 2:
        raise ValueError('not enough unique award_date values for a time split')

    min_train_rows = max(50, min(300, int(len(region_data) * 0.30)))
    min_test_rows = max(20, min(100, int(len(region_data) * 0.10)))

    split_candidates = []
    for cutoff_date in date_counts['award_date'].iloc[1:]:
        train_rows = int((region_data['award_date'] < cutoff_date).sum())
        test_rows = int((region_data['award_date'] >= cutoff_date).sum())
        split_candidates.append({
            'cutoff_date': cutoff_date,
            'train_rows': train_rows,
            'test_rows': test_rows,
            'train_ratio_actual': train_rows / len(region_data)
        })

    split_candidates = pd.DataFrame(split_candidates)
    valid_candidates = split_candidates.loc[
        (split_candidates['train_rows'] >= min_train_rows) &
        (split_candidates['test_rows'] >= min_test_rows)
    ].copy()

    if valid_candidates.empty:
        raise ValueError(
            f'no valid cutoff found with min_train_rows={min_train_rows} and min_test_rows={min_test_rows}'
        )

    valid_candidates['ratio_gap'] = (valid_candidates['train_ratio_actual'] - train_ratio).abs()
    best_split = valid_candidates.sort_values(['ratio_gap', 'cutoff_date']).iloc[0]
    cutoff_date = best_split['cutoff_date']

    train_data = region_data.loc[region_data['award_date'] < cutoff_date].reset_index(drop=True).copy()
    test_data = region_data.loc[region_data['award_date'] >= cutoff_date].reset_index(drop=True).copy()

    split_info = {
        'cutoff_date': cutoff_date,
        'train_rows': int(best_split['train_rows']),
        'test_rows': int(best_split['test_rows']),
        'train_ratio_actual': float(best_split['train_ratio_actual']),
        'min_train_rows': min_train_rows,
        'min_test_rows': min_test_rows,
        'candidate_count': int(len(valid_candidates))
    }

    return train_data, test_data, split_info

data = pd.read_csv(cleaned_data_path)
data['lspe_id'] = data['lspe_id'].astype(str)
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data['award_date'] = pd.to_datetime(data['award_date'], errors='coerce')
data = data.sort_values(['nama_daerah', 'award_date', 'date']).reset_index(drop=True)

region_summary = data.groupby(['nama_daerah', 'lspe_id']).agg(rows=('nama_daerah', 'size'), min_award_date=('award_date', 'min'), max_award_date=('award_date', 'max')).reset_index().sort_values(['nama_daerah', 'lspe_id']).reset_index(drop=True)

print('FULL CLEANED DATA SUMMARY')
display(region_summary)
display(data.head(3))


FULL CLEANED DATA SUMMARY


,nama_daerah,lspe_id,rows,min_award_date,max_award_date
0,Jakarta,127,8678,2014-12-24,2023-10-18
1,Jawa Tengah,42,6038,2017-04-28,2023-10-16
2,Jawa Timur,15,6440,2014-12-19,2017-10-24


,lspe_id,nama_daerah,source_year,source_file,date,buyer_name,tender_title,mainprocurementcategory,tender_minvalue,tender_datepublished,tender_status,award_title,award_date,award_value,award_supplier,days_to_award,budget_utilization_ratio
0,127,Jakarta,2015,ocds-data-tender-2015-127.json,2014-12-24,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,Goods,1.499502e+09,2014-12-08,Complete,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,2014-12-24,1.157888e+09,Cv. Usaha Selaras,16,0.772182
1,127,Jakarta,2015,ocds-data-tender-2015-127.json,2015-04-15,Pemerintah Daerah Provinsi Dki Jakarta,Penyelenggaraan Kegiatan Asia Golf Tourism Con...,Services,1.988607e+10,2015-03-31,Complete,Penyelenggaraan Kegiatan Asia Golf Tourism Con...,2015-04-15,1.934999e+10,Pt. Ekspo Kreatif Indo,15,0.973043
2,127,Jakarta,2015,ocds-data-tender-2015-127.json,2015-04-17,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Naskah Soal Ujian Sekolah/ Madrasah ...,Services,1.612680e+09,2015-04-02,Complete,Pengadaan Naskah Soal Ujian Sekolah/ Madrasah ...,2015-04-17,1.263166e+09,Pt Pura Barutama,15,0.783271


## 3. Train Per Daerah and Save Regional Artifacts

In [3]:
master_regions = {}
master_summary_rows = []
master_train_outputs = []
master_test_outputs = []
master_shap_outputs = []
master_reference_outputs = []
master_demo_inputs = []
master_demo_outputs = []

for region in region_summary.itertuples(index=False):
    nama_daerah = region.nama_daerah
    lspe_id = str(region.lspe_id)
    total_rows = int(region.rows)
    region_key = nama_daerah.strip().casefold()
    region_slug = f'{slug(nama_daerah)}_{lspe_id}'
    region_dir = region_root / region_slug
    region_dir.mkdir(parents=True, exist_ok=True)

    print('=' * 80)
    print(f'TRAINING DAERAH: {nama_daerah} ({lspe_id})')
    print('=' * 80)
    print(f'Total rows available: {total_rows:,}')

    if total_rows < min_rows_per_region:
        print(f'Skipped because rows < {min_rows_per_region}')
        master_summary_rows.append({'lspe_id': lspe_id, 'nama_daerah': nama_daerah, 'status': 'skipped_not_enough_rows', 'total_rows': total_rows, 'artifact_subdir': str(region_dir.relative_to(artifact_root))})
        continue

    region_data = data.loc[data['nama_daerah'] == nama_daerah].copy().sort_values(['award_date', 'date']).reset_index(drop=True)
    region_data = engineer_features(region_data)

    try:
        train_data, test_data, split_info = make_time_based_split(region_data, train_ratio)
    except ValueError as exc:
        print(f'Skipped because time split failed: {exc}')
        master_summary_rows.append({
            'lspe_id': lspe_id,
            'nama_daerah': nama_daerah,
            'status': 'skipped_split_failed',
            'artifact_subdir': str(region_dir.relative_to(artifact_root)),
            'total_rows': len(region_data),
            'split_reason': str(exc)
        })
        continue

    train_max_date = train_data['award_date'].max()
    test_min_date = test_data['award_date'].min()

    print(f'Train rows      : {len(train_data):,}')
    print(f'Test rows       : {len(test_data):,}')
    print(f'Cutoff date     : {split_info["cutoff_date"]}')
    print(f'Actual ratio     : {split_info["train_ratio_actual"]:.3f}')
    print(f'Valid cutoffs    : {split_info["candidate_count"]}')
    print(f'Train max date  : {train_max_date.date()}')
    print(f'Test min date   : {test_min_date.date()}')
    print(f'Leakage check   : {bool(train_max_date <= test_min_date)}')

    if not bool(train_max_date <= test_min_date):
        raise ValueError(
            f'Temporal split failed for {nama_daerah}. '
            f'cutoff_date={split_info["cutoff_date"]}, train_max_date={train_max_date}, test_min_date={test_min_date}'
        )

    feature_columns = numeric_features + categorical_features
    preprocessor = make_preprocessor()
    X_train = preprocessor.fit_transform(train_data[feature_columns])
    X_test = preprocessor.transform(test_data[feature_columns])

    model = IsolationForest(n_estimators=400, contamination=contamination, random_state=random_state, n_jobs=-1)
    model.fit(X_train)
    train_scores = -model.score_samples(X_train)
    test_scores = -model.score_samples(X_test)
    medium_cutoff = float(np.quantile(train_scores, 0.70))
    anomaly_threshold = float(np.quantile(train_scores, 1 - contamination))

    print(f'Medium cutoff   : {medium_cutoff:.6f}')
    print(f'Anomaly cutoff  : {anomaly_threshold:.6f}')

    train_output = make_scored_output(train_data, train_scores, medium_cutoff, anomaly_threshold)
    test_output = make_scored_output(test_data, test_scores, medium_cutoff, anomaly_threshold)

    print('\nTrain label distribution:')
    print(train_output['prediction_label'].value_counts().to_string())
    print('\nTest label distribution:')
    print(test_output['prediction_label'].value_counts().to_string())

    explainer_shap = shap.TreeExplainer(model)
    shap_values_test = explainer_shap.shap_values(X_test)
    if isinstance(shap_values_test, list):
        shap_values_test = shap_values_test[0]
    feature_names_clean = clean_feature_names(preprocessor.get_feature_names_out())
    test_output = build_local_explanations(test_output, shap_values_test, feature_names_clean)

    shap_importance = pd.DataFrame({'feature': feature_names_clean, 'mean_abs_shap': np.abs(shap_values_test).mean(axis=0)})
    shap_importance['raw_feature'] = shap_importance['feature'].apply(lambda feature_name: 'mainprocurementcategory' if str(feature_name).startswith('cat_') else feature_name)
    shap_global = shap_importance.groupby('raw_feature', as_index=False)['mean_abs_shap'].sum().sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    shap_global.insert(0, 'lspe_id', lspe_id)
    shap_global.insert(1, 'nama_daerah', nama_daerah)

    print('\nTop SHAP features:')
    display(shap_global.head(10))

    reference_thresholds, category_reference = build_reference_thresholds(train_output)
    category_reference.insert(0, 'lspe_id', lspe_id)
    category_reference.insert(1, 'nama_daerah', nama_daerah)
    print('Reference thresholds by category:')
    display(category_reference.head(10))

    demo_cases = select_demo_cases(test_output)
    if not demo_cases.empty:
        demo_cases['demo_case_label'] = [f'{region_slug}_{label}_demo' for label in demo_cases['prediction_label'].tolist()]
        print('Demo cases selected:')
        display(demo_cases[['demo_case_label', 'prediction_label', 'anomaly_score', 'nama_daerah']])
    else:
        print('No demo cases selected for this daerah.')

    train_csv_path = cleaned_train_dir / f'train_data_{lspe_id}.csv'
    test_csv_path = cleaned_test_dir / f'test_data_{lspe_id}.csv'
    train_data[base_export_columns].to_csv(train_csv_path, index=False)
    test_data[base_export_columns].to_csv(test_csv_path, index=False)
    train_data.to_csv(region_dir / 'train_post_award_split.csv', index=False)
    test_data.to_csv(region_dir / 'test_post_award_split.csv', index=False)
    train_output.to_csv(region_dir / 'train_predictions.csv', index=False)
    test_output.to_csv(region_dir / 'test_predictions_with_explanations.csv', index=False)
    shap_global.to_csv(region_dir / 'shap_global_importance.csv', index=False)
    category_reference.to_csv(region_dir / 'reference_thresholds_by_category.csv', index=False)
    if not demo_cases.empty:
        demo_cases[base_export_columns + ['demo_case_label']].to_csv(region_dir / 'demo_input_cases.csv', index=False)
        demo_cases.to_csv(region_dir / 'demo_expected_outputs.csv', index=False)

    joblib.dump(model, region_dir / 'isolation_forest.joblib')
    joblib.dump(preprocessor, region_dir / 'preprocessor.joblib')
    joblib.dump(explainer_shap, region_dir / 'shap_explainer.joblib')

    model_config = {'model_type': 'IsolationForest', 'routing_mode': 'per_nama_daerah', 'routing_key': region_key, 'region_name': nama_daerah, 'lspe_id': lspe_id, 'numeric_features': numeric_features, 'categorical_features': categorical_features, 'feature_columns': feature_columns, 'contamination': contamination, 'medium_cutoff': medium_cutoff, 'anomaly_threshold': anomaly_threshold, 'train_rows': len(train_output), 'test_rows': len(test_output), 'train_end_date': train_max_date, 'test_start_date': test_min_date}
    explanation_meta = {'region_name': nama_daerah, 'lspe_id': lspe_id, 'feature_names_preprocessed': list(feature_names_clean), 'categorical_features': categorical_features, 'routing_note': 'Master bundle selects the daerah model package using nama_daerah.', 'shap_note': 'Positive SHAP raises anomaly score; negative SHAP lowers anomaly score.'}
    (region_dir / 'model_config.json').write_text(json.dumps(to_builtin(model_config), indent=2), encoding='utf-8')
    (region_dir / 'explanation_meta.json').write_text(json.dumps(to_builtin(explanation_meta), indent=2), encoding='utf-8')
    (region_dir / 'reference_thresholds.json').write_text(json.dumps(to_builtin(reference_thresholds), indent=2), encoding='utf-8')

    master_regions[region_key] = {'lspe_id': lspe_id, 'nama_daerah': nama_daerah, 'routing_key': region_key, 'model': model, 'preprocessor': preprocessor, 'explainer': explainer_shap, 'model_config': model_config, 'explanation_meta': explanation_meta, 'reference_thresholds': reference_thresholds, 'artifact_dir': str(region_dir.resolve())}
    master_summary_rows.append({'lspe_id': lspe_id, 'nama_daerah': nama_daerah, 'status': 'trained', 'artifact_subdir': str(region_dir.relative_to(artifact_root)), 'total_rows': len(region_data), 'train_rows': len(train_output), 'test_rows': len(test_output), 'train_ratio_actual': split_info['train_ratio_actual'], 'cutoff_date': split_info['cutoff_date'], 'train_end_date': train_max_date, 'test_start_date': test_min_date, 'medium_cutoff': medium_cutoff, 'anomaly_threshold': anomaly_threshold, 'train_anomaly_rate': float(train_output['anomaly_flag'].mean()), 'test_anomaly_rate': float(test_output['anomaly_flag'].mean())})
    master_train_outputs.append(train_output)
    master_test_outputs.append(test_output)
    master_shap_outputs.append(shap_global)
    master_reference_outputs.append(category_reference)
    if not demo_cases.empty:
        master_demo_inputs.append(demo_cases[base_export_columns + ['demo_case_label']])
        master_demo_outputs.append(demo_cases)

    print(f'Artifacts saved to: {region_dir.resolve()}')


TRAINING DAERAH: Jakarta (127)
Total rows available: 8,678
Train rows      : 7,376
Test rows       : 1,302
Cutoff date     : 2022-03-24 00:00:00
Actual ratio     : 0.850
Valid cutoffs    : 1424
Train max date  : 2022-03-23
Test min date   : 2022-03-24
Leakage check   : True
Medium cutoff   : 0.462116
Anomaly cutoff  : 0.581943

Train label distribution:
prediction_label
normal     5163
warning    1991
anomaly     222

Test label distribution:
prediction_label
normal     961
warning    304
anomaly     37

Top SHAP features:


,lspe_id,nama_daerah,raw_feature,mean_abs_shap
0,127,Jakarta,mainprocurementcategory,0.462451
1,127,Jakarta,award_quarter,0.196100
2,127,Jakarta,value_gap,0.191323
3,127,Jakarta,award_month,0.158950
4,127,Jakarta,tender_minvalue,0.157495
5,127,Jakarta,award_title_word_count,0.149011
6,127,Jakarta,title_word_count,0.142999
7,127,Jakarta,award_value,0.142364
8,127,Jakarta,award_value_per_day,0.136477
9,127,Jakarta,log_tender_minvalue,0.129406


Reference thresholds by category:


,lspe_id,nama_daerah,mainprocurementcategory,award_value_p99,days_to_award_p95,budget_ratio_p05,budget_ratio_p95
0,127,Jakarta,Goods,3.133035e+10,34.0,0.625057,0.996095
1,127,Jakarta,Services,1.481400e+10,65.0,0.690826,0.989570
2,127,Jakarta,Works,2.431357e+11,49.2,0.780879,0.979158


Demo cases selected:


...

Reference thresholds by category:


,lspe_id,nama_daerah,mainprocurementcategory,award_value_p99,days_to_award_p95,budget_ratio_p05,budget_ratio_p95
0,15,Jawa Timur,Goods,2.290966e+10,28.0,0.708153,0.995284
1,15,Jawa Timur,Services,3.527593e+09,53.0,0.796657,0.996892
2,15,Jawa Timur,Works,5.195415e+10,36.6,0.768857,0.987327


Demo cases selected:


,demo_case_label,prediction_label,anomaly_score,nama_daerah
0,jawa_timur_15_normal_demo,normal,0.440403,Jawa Timur
1,jawa_timur_15_warning_demo,warning,0.491479,Jawa Timur
2,jawa_timur_15_anomaly_demo,anomaly,0.666770,Jawa Timur


Artifacts saved to: C:\Users\VICTUS\coding\collab\03_training\artifacts\post_award_anomaly\by_daerah\jawa_timur_15


## 4. Build One Master Joblib and Final Summary

In [4]:
if not master_regions:
    raise RuntimeError('No daerah model was trained. Check cleaned data and min_rows_per_region.')

master_summary = pd.DataFrame(master_summary_rows).sort_values(['status', 'nama_daerah']).reset_index(drop=True)
master_summary.to_csv(artifact_root / 'master_training_summary.csv', index=False)

if master_train_outputs:
    pd.concat(master_train_outputs, ignore_index=True).to_csv(artifact_root / 'master_train_predictions.csv', index=False)
if master_test_outputs:
    master_test_predictions = pd.concat(master_test_outputs, ignore_index=True)
    master_test_predictions.to_csv(artifact_root / 'master_test_predictions_with_explanations.csv', index=False)
else:
    master_test_predictions = pd.DataFrame()
if master_shap_outputs:
    master_shap_global = pd.concat(master_shap_outputs, ignore_index=True)
    master_shap_global.to_csv(artifact_root / 'master_shap_global_importance.csv', index=False)
    master_shap_aggregated = master_shap_global.groupby('raw_feature', as_index=False)['mean_abs_shap'].mean().sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    master_shap_aggregated.to_csv(artifact_root / 'master_shap_global_importance_aggregated.csv', index=False)
else:
    master_shap_aggregated = pd.DataFrame(columns=['raw_feature', 'mean_abs_shap'])
if master_reference_outputs:
    pd.concat(master_reference_outputs, ignore_index=True).to_csv(artifact_root / 'master_reference_thresholds_by_category.csv', index=False)
if master_demo_inputs:
    pd.concat(master_demo_inputs, ignore_index=True).to_csv(artifact_root / 'master_demo_input_cases.csv', index=False)
if master_demo_outputs:
    pd.concat(master_demo_outputs, ignore_index=True).to_csv(artifact_root / 'master_demo_expected_outputs.csv', index=False)

master_bundle = {
    'bundle_type': 'multi_daerah_isolation_forest_router',
    'routing_column': 'nama_daerah',
    'routing_strategy': 'casefold_exact_match',
    'shared_feature_contract': {'numeric_features': numeric_features, 'categorical_features': categorical_features, 'base_export_columns': base_export_columns},
    'region_count': len(master_regions),
    'regions': master_regions,
    'training_summary': master_summary.to_dict(orient='records')
}
joblib.dump(master_bundle, master_bundle_path)

master_manifest = {
    'bundle_type': master_bundle['bundle_type'],
    'routing_column': master_bundle['routing_column'],
    'routing_strategy': master_bundle['routing_strategy'],
    'artifact_root': str(artifact_root.resolve()),
    'region_count': len(master_regions),
    'regions': [{'routing_key': key, 'nama_daerah': bundle['nama_daerah'], 'lspe_id': bundle['lspe_id'], 'artifact_dir': bundle['artifact_dir'], 'anomaly_threshold': bundle['model_config']['anomaly_threshold'], 'medium_cutoff': bundle['model_config']['medium_cutoff']} for key, bundle in master_regions.items()]
}
master_manifest_path.write_text(json.dumps(to_builtin(master_manifest), indent=2), encoding='utf-8')

print('MASTER TRAINING SUMMARY')
print('=' * 80)
display(master_summary)

if not master_test_predictions.empty:
    print('Combined test label distribution:')
    print(master_test_predictions['prediction_label'].value_counts().to_string())

print('\nTop aggregated SHAP features across daerah:')
display(master_shap_aggregated.head(10))

print('Saved files:')
print(f'- Master bundle   : {master_bundle_path.resolve()}')
print(f'- Master manifest : {master_manifest_path.resolve()}')
print(f'- Summary csv     : {(artifact_root / 'master_training_summary.csv').resolve()}')
print(f'- Artifact root   : {artifact_root.resolve()}')
print('\nThis final single joblib contains all daerah models inside one package.')


MASTER TRAINING SUMMARY


,lspe_id,nama_daerah,status,artifact_subdir,total_rows,train_rows,test_rows,train_ratio_actual,cutoff_date,train_end_date,test_start_date,medium_cutoff,anomaly_threshold,train_anomaly_rate,test_anomaly_rate
0,127,Jakarta,trained,by_daerah\jakarta_127,8678,7376,1302,0.849965,2022-03-24,2022-03-23,2022-03-24,0.462116,0.581943,0.030098,0.028418
1,42,Jawa Tengah,trained,by_daerah\jawa_tengah_42,6038,3410,100,0.564757,2023-05-04,2023-05-03,2023-05-04,0.458017,0.587666,0.030205,0.020000
2,15,Jawa Timur,trained,by_daerah\jawa_timur_15,6440,2986,100,0.463665,2017-09-08,2017-09-07,2017-09-08,0.469430,0.573563,0.030141,0.060000


Combined test label distribution:
prediction_label
normal     1111
warning     346
anomaly      45

Top aggregated SHAP features across daerah:


,raw_feature,mean_abs_shap
0,mainprocurementcategory,0.470322
1,value_gap,0.176655
2,award_quarter,0.175311
3,tender_minvalue,0.158695
4,award_month,0.152082
5,award_value_per_day,0.148464
6,award_value,0.142634
7,days_to_award,0.142432
8,log_tender_minvalue,0.137897
9,log_award_value,0.127498


Saved files:
- Master bundle   : C:\Users\VICTUS\coding\collab\03_training\artifacts\post_award_anomaly\master_model.joblib
- Master manifest : C:\Users\VICTUS\coding\collab\03_training\artifacts\post_award_anomaly\master_model_manifest.json
- Summary csv     : C:\Users\VICTUS\coding\collab\03_training\artifacts\post_award_anomaly\master_training_summary.csv
- Artifact root   : C:\Users\VICTUS\coding\collab\03_training\artifacts\post_award_anomaly

This final single joblib contains all daerah models inside one package.
